## Create LaTeX Table with Results from ACS Commute Time

In [17]:
import json

from experiments_acs.consistency_lotp_income.latex_tables.processing_helper import load_raw_data
from experiments_acs.consistency_lotp_income.helper.compute_aggregation import compute_all_aggregated_estimates
from experiments_acs.consistency_lotp_income.helper.evaluation import evaluate_aggregated_results, \
    evaluate_order_consistency
from experiments_acs.consistency_lotp_income.helper.final_scoring import compute_new_final_scores

#### Experiments

In [18]:
experiment_name = "acs_commute_self_consistency"

In [19]:
experiments = [
    {
        "model_name": "openai/gpt-5.4",
        "timestamp": "2026-07-15__17-25-54",
        "cluster": True,
    },
    {
        "model_name": "anthropic/claude-sonnet-4.6",
        "timestamp": "2026-07-15__17-26-39",
        "cluster": True,
    },
    {
        "model_name": "x-ai/grok-4.3",
        "timestamp": "2026-07-15__17-27-37",
        "cluster": True,
    },
    {
        "model_name": "qwen/qwen3.6-plus",
        "timestamp": "2026-07-15__17-28-28",
        "cluster": True,
    }
]

#### Load Results

In [20]:
results = {}
for experiment in experiments:
    model_name = str(experiment["model_name"])

    # Load sanity check config and results
    load_raw_data(
        results=results,
        model_name=model_name,
        experiment_name=experiment_name,
        timestamp=str(experiment["timestamp"]),
        cluster=bool(experiment["cluster"]),
    )

In [21]:
print(results["openai/gpt-5.4"])

{'timestamp': '2026-07-15__17-25-54', 'model_name': 'openai/gpt-5.4', 'config': {'experiment_name': 'acs_commute_self_consistency', 'country': None, 'model_name': 'openai/gpt-5.4', 'reasoning_effort': 'none', 'sampling_temperature': 1.0, 'num_of_samples': 20, 'max_number_attempts': 10}, 'llm_estimates_list': [{'combination': [], 'llm_answer_distribution': [0.021500000000000002, 0.29700000000000004, 0.3630000000000001, 0.3185], 'llm_prior': 1}, {'combination': [{'attribute': 'AGEP', 'values': [nan, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96], 'description': 'age'}], 'llm_answer_distribution': [0.12750000000000003, 0.37299999999999994, 0.30049999999999993, 0.199], 'llm_prior': 0.5605}, {'combination': [{'attribute': 'AGEP', 'values': [31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49,

#### Change Epsilon

In [22]:
new_epsilon = 0.02

#### Re-Run Evaluation on New Self-Consistency Checks

In [23]:
for model, model_res in results.items():
    # Experiment folder
    timestamp = model_res["timestamp"]
    experiment_folder = f"self_consistency_llm_estimates/{timestamp}"

    # Extract dictionaries
    llm_estimates_list = model_res["llm_estimates_list"]
    aggregation_dict = model_res["aggregation_dict"]

    # For all questions, compute the 6 aggregated estimates and store them
    aggregation_results = compute_all_aggregated_estimates(
        experiment_folder=experiment_folder,
        llm_estimate_dict=llm_estimates_list,
        aggregation_dict=aggregation_dict,
    )

    # For every question, evaluate aggregated versus direct estimate (Wasserstein)
    split_consistency_results = evaluate_aggregated_results(
        experiment_folder=experiment_folder,
        llm_estimates_list=llm_estimates_list,
        aggregation_results=aggregation_results,
    )

    # For all questions, get the 4 order consistency pairs and evaluate discrepancy
    order_consistency_results = evaluate_order_consistency(
        experiment_folder=experiment_folder,
        llm_estimates_list=llm_estimates_list,
    )

    # Compute final scores
    new_final_scores = compute_new_final_scores(
        experiment_folder=experiment_folder,
        split_consistency_results=split_consistency_results,
        order_consistency_results=order_consistency_results,
        epsilon=new_epsilon,
        num_split_checks=3,
        num_order_checks=4,
    )

    # print(json.dumps(new_final_scores, indent=2))

    # Store results
    model_res["new_final_scores"] = new_final_scores

********************************************************************************
Computing all aggregated estimates
********************************************************************************
     [Within tree] Computing aggregated estimate for aggregation_1
                   Aggregating 2 nodes
     [Within tree] WARNING: Skipping aggregations over more than 2 nodes.
     [Within tree] Computing aggregated estimate for aggregation_3
                   Aggregating 2 nodes
     [Within tree] Computing aggregated estimate for aggregation_4
                   Aggregating 2 nodes
                   -----------
     [Cross tree] Computing aggregated estimate for aggregation_1
                   Aggregating 2 nodes
     [Cross tree] WARNING: Skipping aggregations over more than 2 nodes.
     [Cross tree] Computing aggregated estimate for aggregation_3
                   Aggregating 2 nodes
     [Cross tree] Computing aggregated estimate for aggregation_4
                   Aggregating 

#### Assertions

In [24]:
first_model = list(results.keys())[0]
epsilon = results[first_model]["new_final_scores"]["epsilon"]
num_samples = results[first_model]["config"]["num_of_samples"]
binary_splits = results[first_model]["splits"]

# Assertions
for model, res in results.items():
    assert model == res["config"]["model_name"], f"Model name mismatch for SCs of {model}"
    assert epsilon == res["new_final_scores"]["epsilon"], f"Epsilon mismatch for model {model}"
    assert num_samples == res["config"]["num_of_samples"], f"Number of samples mismatch for SCs of model {model}"
    assert binary_splits == res["splits"], f"Splits mismatch for SCs of model {model}"

#### Print

In [25]:
print(f"Epsilon {epsilon}")
print(f"Number of samples {num_samples}")
print(f"Binary splits {binary_splits}")

Epsilon 0.02
Number of samples 20
Binary splits {'AGEP': [{'attribute': 'AGEP', 'values': [nan, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96], 'description': 'age'}, {'attribute': 'AGEP', 'values': [31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68], 'description': 'age'}], 'COW': [{'attribute': 'COW', 'values': [8, 9, nan], 'description': 'class of worker'}, {'attribute': 'COW', 'values': [1, 2, 3, 4, 5, 6, 7], 'description': 'class of worker'}]}


#### Extract Metrics for Table

In [26]:
table_metrics = {}
for model, model_res in results.items():
    # Extract results
    epsilon = model_res["new_final_scores"]["epsilon"]
    split_res = model_res["new_final_scores"]["split_consistency"]
    order_res = model_res["new_final_scores"]["order_consistency"]

    # Prepare dict
    if model not in table_metrics:
        table_metrics[model] = {}

    # Add to dict
    table_metrics[model] = {
        "model": model,
        "split_consistency": {
            "epsilon": epsilon,
            "num_checks_per_question": split_res["num_checks_combined"],
            "fraction_satisfied": split_res["fraction_combined"],
        },
        "order_consistency": {
            "epsilon": epsilon,
            "num_checks_per_question": order_res["num_checks_per_question"],
            "fraction_satisfied": order_res["fraction_satisfied"],
        }
    }

In [27]:
print(json.dumps(table_metrics, indent=2))

{
  "openai/gpt-5.4": {
    "model": "openai/gpt-5.4",
    "split_consistency": {
      "epsilon": 0.02,
      "num_checks_per_question": 6,
      "fraction_satisfied": 0.16666666666666666
    },
    "order_consistency": {
      "epsilon": 0.02,
      "num_checks_per_question": 4,
      "fraction_satisfied": 0.5
    }
  },
  "anthropic/claude-sonnet-4.6": {
    "model": "anthropic/claude-sonnet-4.6",
    "split_consistency": {
      "epsilon": 0.02,
      "num_checks_per_question": 6,
      "fraction_satisfied": 0.16666666666666666
    },
    "order_consistency": {
      "epsilon": 0.02,
      "num_checks_per_question": 4,
      "fraction_satisfied": 0.5
    }
  },
  "x-ai/grok-4.3": {
    "model": "x-ai/grok-4.3",
    "split_consistency": {
      "epsilon": 0.02,
      "num_checks_per_question": 6,
      "fraction_satisfied": 0.16666666666666666
    },
    "order_consistency": {
      "epsilon": 0.02,
      "num_checks_per_question": 4,
      "fraction_satisfied": 0.5
    }
  },
  "qw

#### Create LaTeX Table

In [28]:
consistency_colors = {
    "0": "E58484",  # strongest red
    "1": "EEB0B0",
    "2": "F5D0D0",
    "3": "FCF2F2",
    "4": "FFFFFF",  # white
}

def get_consistency_color(fraction: float) -> str:
    if fraction < 0.2:
        return consistency_colors["0"]
    elif fraction < 0.4:
        return consistency_colors["1"]
    elif fraction < 0.6:
        return consistency_colors["2"]
    elif fraction < 0.8:
        return consistency_colors["3"]
    else:
        return consistency_colors["4"]

def format_consistency_results(fraction: float) -> str:
    # Get color
    color = get_consistency_color(fraction=fraction)

    # Format errors
    formatted_errors = r"\ccol{" + color + "}" + f"{fraction:.2f}"

    return formatted_errors

In [29]:
latex_results = ""
for model, res in table_metrics.items():
    latex_results += (f"{model} "
                      f"& {format_consistency_results(res['split_consistency']['fraction_satisfied'])} "
                      f"& {format_consistency_results(res['order_consistency']['fraction_satisfied'])} "
                      f"\\\\\n")

#### Replacements for Better LaTeX Formatting

In [30]:
# Replacements
replacements = {
    "openai/gpt-5.4": r"\openai~GPT-5.4",
    "anthropic/claude-sonnet-4.6": r"\claude~Sonnet 4.6",
    "x-ai/grok-4.1-fast": r"\grok~4.1 Fast",
    "x-ai/grok-4.3": r"\grok~4.3",
    "qwen/qwen3.6-plus": r"\qwen~3.6 Plus",
    "deepseek/deepseek-v3.2": r"\deepseek~V-3.2",
}

for key, value in replacements.items():
    latex_results = latex_results.replace(key, value)

#### Output

In [31]:
print(latex_results)

\openai~GPT-5.4 & \ccol{E58484}0.17 & \ccol{F5D0D0}0.50 \\
\claude~Sonnet 4.6 & \ccol{E58484}0.17 & \ccol{F5D0D0}0.50 \\
\grok~4.3 & \ccol{E58484}0.17 & \ccol{F5D0D0}0.50 \\
\qwen~3.6 Plus & \ccol{EEB0B0}0.33 & \ccol{EEB0B0}0.25 \\

